In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita a exibição inline de gráficos gerados pelo matplotlib no notebook
%matplotlib inline

# Encontrar conjuntos de dados com a API EEGDash

**Dificuldade 1** | **Tempo de execução: <2m** | **Computação: CPU**

O EEGDash disponibiliza um índice de metadados com centenas de datasets de EEG organizados no formato BIDS,
servidos pela API pública REST em https://data.eegdash.org. O mesmo catálogo abastece o [NEMAR](https://nemar.org),
o portal do ecossistema EEGLAB que hospeda datasets de EEG/MEG com ferramentas de navegação, computação e proveniência
(Delorme et al., 2022). O cliente :class:`~eegdash.api.EEGDash` busca, filtra e resume esse índice sem precisar
baixar uma única amostra de sinal.

Palavras-chave: carregamento, metadados, API


## Objetivos de Aprendizagem
- Inicializar um cliente :class:`~eegdash.api.EEGDash` e explorar seus métodos públicos.
- Buscar registros utilizando múltiplos filtros (entidades BIDS + metadados científicos).
- Converter os resultados das consultas em um :class:`pandas.DataFrame` para análise.
- Calcular estatísticas da coorte (sujeitos, frequências de amostragem, modalidades) apenas a partir de metadados.


## Requisitos
- Cerca de 1 minuto em CPU; consome apenas metadados (< 5 MB de rede).
- Conexão de rede necessária; o ``EEGDash`` consome a API pública em
  ``https://data.eegdash.org``. Nenhuma autenticação é necessária para leitura.


Configuração inicial. Como não há processos estocásticos, não é necessário definir seed.


In [ ]:
# Importação das bibliotecas essenciais de visualização e manipulação de dados
import matplotlib.pyplot as plt
import pandas as pd

# Importação do cliente principal do EEGDash e funções auxiliares de estilização gráfica
import eegdash
from eegdash import EEGDash
from eegdash.viz import EEGDASH_BLUE, style_figure, use_eegdash_style

# Aplicação do estilo visual padronizado do EEGDash para os gráficos
use_eegdash_style()
# Exibe a versão instalada do eegdash
print(f"eegdash {eegdash.__version__}")

## Passo 1: Descobrir os métodos do cliente EEGDash
``EEGDash()`` encapsula o endpoint REST. Executar ``dir()`` costuma ser mais rápido do que consultar a documentação.

**Previsão:** Um cliente de catálogo somente leitura deve expor pelo menos três verbos: count, find, get. Quantos o ``EEGDash`` realmente possui?

**Execução:** Imprima a lista de métodos públicos.


In [ ]:
# Inicializa a instância do cliente da API EEGDash
client = EEGDash()
# Obtém e filtra todos os métodos públicos (que não começam com sublinhado e são chamáveis)
public_methods = sorted(
    m for m in dir(client) if not m.startswith("_") and callable(getattr(client, m))
)
print(f"EEGDash() expõe {len(public_methods)} métodos públicos:")
for m in public_methods:
    print(f"  - {m}")

**Análise:** Temos ``count``, ``exists``, ``find``, ``find_one``,
``find_datasets``, ``search_datasets``, ``get_dataset``, além de métodos administrativos.
``find`` e ``find_datasets`` são os principais métodos de leitura utilizados ao longo dos tutoriais.


## Valide seu resultado
Uma inicialização bem-sucedida de ``EEGDash()`` e a execução de ``count()`` devem apresentar:

1. Uma lista de métodos contendo ``find``, ``count`` e ``get_dataset``.
2. Uma contagem de registros superior a 10.000 (índice em constante crescimento).
3. Uma projeção com colunas como ``subject``, ``session`` e ``task``.
4. Resultados filtrados contendo colunas adicionais como ``sfreq`` e ``n_channels``.


## Passo 2: Qual é o tamanho do catálogo?
O método ``count`` é executado no lado do servidor e representa a requisição mais leve possível.


In [ ]:
# Executa a contagem total de registros no índice da API passando um filtro vazio
n_records = client.count({})
# Exibe a contagem total formatada com separadores de milhar
print(f"registros no índice: {n_records:,}")

## Passo 3: Uma varredura ampla vs. uma consulta filtrada
Uma chamada não filtrada a ``find({})`` retorna uma projeção simplificada: apenas entidades BIDS, sem propriedades físicas do sinal.
Já a filtragem por dataset desbloqueia os metadados completos (frequência de amostragem, número de canais, duração em amostras).

**Execute** ambas as opções e compare as colunas resultantes.


In [ ]:
# Consulta ampla sem filtros, limitada a 200 registros
broad = pd.DataFrame(client.find({}, limit=200)).drop(columns=["_id"], errors="ignore")
# Consulta direcionada a uma lista específica de datasets via operador MongoDB $in
focused = pd.DataFrame(
    client.find(
        {"dataset": {"$in": ["ds002718", "ds005514", "ds005863", "ds003061"]}},
        limit=200,
    )
).drop(columns=["_id"], errors="ignore")
# Exibe as dimensões (linhas x colunas) de cada DataFrame
print(f"ampla    : {broad.shape[0]} linhas x {broad.shape[1]} colunas")
print(f"focada   : {focused.shape[0]} linhas x {focused.shape[1]} colunas")
# Identifica as colunas extras que surgem ao especificar o dataset
extra = sorted(set(focused.columns) - set(broad.columns))
print(f"colunas adicionais quando filtrado: {extra}")

## Passo 4: O que um registro realmente contém?
Um registro é um *documento de metadados* (uma linha por arquivo BIDS), e não o sinal bruto em si.
Os campos cobrem entidades BIDS (sujeito, tarefa, sessão, corrida/run), metadados científicos (frequência de amostragem,
quantidade de canais, duração em amostras) e informações de armazenamento. Conforme a especificação EEG-BIDS
(Pernet et al. 2019), cada gravação disponibiliza esses atributos.


In [ ]:
# Define as principais colunas de interesse para inspeção
fields = [
    "dataset",
    "subject",
    "task",
    "session",
    "run",
    "sampling_frequency",
    "nchans",
    "ntimes",
    "datatype",
]
# Seleciona a primeira linha do DataFrame focado como amostra
sample = focused.iloc[0]
# Converte a série selecionada em formato tabular vertical
record_view = sample[fields].to_frame("valor")
# Calcula e adiciona a duração em segundos a partir do número de amostras e da taxa de amostragem
record_view.loc["duração (s)"] = round(
    sample["ntimes"] / sample["sampling_frequency"], 1
)
record_view

## Passo 5: Análise da coorte utilizando o DataFrame
Métodos padrão do pandas como ``value_counts``, ``groupby`` e ``describe`` resolvem a maior parte das dúvidas sobre o catálogo;
o cliente do EEGDash serve apenas como meio de obtenção dos dados.


Registros por conjunto de dados (dataset).


In [ ]:
# Contagem de registros individuais associados a cada identificador de dataset
focused["dataset"].value_counts().to_frame("registros")

Distribuição da frequência de amostragem.


In [ ]:
# Identifica as 8 frequências de amostragem mais comuns na amostra obtida
focused["sampling_frequency"].value_counts().head(8).to_frame("registros")

Principais tarefas (tasks).


In [ ]:
# Lista as 8 tarefas experimentais mais frequentes
focused["task"].value_counts().head(8).to_frame("registros")

## Passo 6: Visualizar a coorte
Um gráfico de barras horizontais de "registros por dataset" permite uma leitura imediata.


In [ ]:
# Ordena a contagem de registros em ordem crescente para exibição em barras horizontais
counts = focused["dataset"].value_counts().iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.2))
# Cria as barras horizontais com a cor padrão do EEGDash
bars = ax.barh(counts.index, counts.values, color=EEGDASH_BLUE)
# Adiciona o rótulo numérico ao lado de cada barra
ax.bar_label(bars, padding=4, fontsize=9, color="#102A43")
ax.set_xlabel("registros (n)")
ax.set_ylabel("conjunto de dados (dataset)")
ax.set_xlim(0, counts.max() * 1.15)
# Aplica título, subtítulo e indicação de fonte padronizados na figura
style_figure(
    fig,
    title="Registros por conjunto de dados",
    subtitle=f"{len(focused)} registros | 4 datasets consultados via $in",
    source="EEGDash plot_00 | fonte: data.eegdash.org",
)
plt.show()

## Passo 7: Documentos a nível de dataset
``find_datasets`` retorna os documentos de *catálogo* (um por conjunto de dados completo),
contendo DOI, versão BIDS, demografia e citações. Vamos convertê-los em um DataFrame
e filtrar os datasets com maior número de sujeitos.


In [ ]:
# Busca até 300 documentos no nível de dataset e converte para DataFrame
catalogue = pd.DataFrame(client.find_datasets({}, limit=300))
# Extrai com segurança a contagem de sujeitos do dicionário de demografia
catalogue["n_subjects"] = catalogue["demographics"].apply(
    lambda d: (d or {}).get("subjects_count", 0) if isinstance(d, dict) else 0
)
# Filtra datasets com pelo menos 5 sujeitos e seleciona o top 10 com mais participantes
shortlist = (
    catalogue.loc[catalogue["n_subjects"] >= 5, ["dataset_id", "n_subjects", "license"]]
    .sort_values("n_subjects", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
shortlist

## Um erro comum e como contorná-lo
**Execução:** Passar um nome de tarefa inexistente retorna uma lista vazia, e não um erro.
A forma de contornar isso é verificar quais tarefas realmente existem no catálogo antes de consultar.


In [ ]:
# Consulta intencional por uma tarefa inexistente no banco de dados
unknown = client.find(task="FacePerceptionXYZ", limit=5)
print(f"registros encontrados para tarefa inexistente: {len(unknown)}")
# Lista as primeiras 8 tarefas válidas presentes nos metadados obtidos
known_tasks = sorted(focused["task"].dropna().unique())[:8]
print(f"tarefas conhecidas (primeiras 8): {known_tasks}")

## Modifique
**Sua vez:** Combine filtros: experimente ``client.find(task="RestingState", sampling_frequency=250, limit=20)``
e depois adicione ``subject="012"``. A linguagem de consulta aceita argumentos nomeados simples ou operadores no formato MongoDB (como ``{"$gte": 250}``).


In [ ]:
# Exemplo de consulta combinando filtro de tarefa específica
combined = client.find(task="RestingState", limit=20)
print(f"registros para task=RestingState: {len(combined)}")

## Construa
**Mini-projeto:** Crie uma coorte candidata: defina uma faixa de taxa de amostragem e um número mínimo de sujeitos,
retornando um DataFrame com ``[dataset_id, n_subjects, sampling_rates_seen]``. Código inicial abaixo.


In [ ]:
def candidate_cohort(min_subjects: int = 5, sfreq_min: float = 200.0) -> pd.DataFrame:
    """Filtra datasets de EEG com >= ``min_subjects`` e ao menos uma taxa de amostragem >= ``sfreq_min``."""
    # Agrupa as frequências de amostragem encontradas por dataset na amostra focada
    rates_per_dataset = (
        focused.groupby("dataset")["sampling_frequency"].agg(set).rename("rates")
    )
    # Faz o merge com os metadados gerais do catálogo
    out = catalogue.merge(
        rates_per_dataset,
        left_on="dataset_id",
        right_index=True,
        how="left",
    )
    # Calcula a frequência máxima de amostragem observada por dataset
    out["max_sfreq"] = out["rates"].apply(
        lambda s: max(s) if isinstance(s, set) and s else 0.0
    )
    # Retorna os 10 datasets mais adequados conforme os critérios estabelecidos
    return (
        out.loc[
            (out["n_subjects"] >= min_subjects) & (out["max_sfreq"] >= sfreq_min),
            ["dataset_id", "n_subjects", "max_sfreq"],
        ]
        .sort_values(["n_subjects", "max_sfreq"], ascending=[False, False])
        .head(10)
    )


# Executa a função do mini-projeto
candidate_cohort(min_subjects=5, sfreq_min=200.0)

## Resultado
Analisamos o índice do EEGDash sem baixar um único arquivo de sinal bruto, executamos consultas com múltiplos campos e selecionamos datasets candidatos em um DataFrame.


## Próximos Passos
A seguir: ``plot_01_first_recording.ipynb`` realiza o download de uma gravação real selecionada na lista acima e inspeciona o sinal bruto, canais e duração.


## Experimente você mesmo
- Substitua ``client.count({})`` por ``client.count({"datatype": "eeg"})``.
- Agrupe ``df`` por ``dataset`` e calcule ``ntimes.sum() / sampling_frequency`` para obter o total de horas registradas por dataset.
- Exporte sua lista de candidatos com ``shortlist.to_parquet("candidates.parquet")``.


## Referências
Consulte :doc:`/references` para a bibliografia completa dos artigos citados.
